In [1]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "kano2016nasal")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "2016_phybeh_edited.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)


df['study_id']="kano2016nasal"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
df.rename(columns={"subject": "ape"}, inplace=True)


In [3]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
df= df.merge(apedf,left_on='ape', right_on='name', how='left')


In [4]:
import re
replace_1=re.compile('(\ |\*)')
df.columns = df.columns.str.replace(replace_1, '_')


In [5]:
# df.columns
df.rename(columns={"ape": "participant",
                "second_playback__no_sound":"second_playback_no_sound"}, inplace=True)


In [6]:
df=df[['study_id','participant', 'sex', 'species', 'first_playback_chimp_scream',
       'first_playback_orangutan_long_call', 'first_playback_no_sound',
       'second_playback_chimp_scream', 'second_playback_orangutan_long_call',
       'second_playback_no_sound']]

data_list = df.values.tolist()
column_name = df.columns.values.tolist()
output = []

for line in data_list:
    for value,name in zip(line[4:],column_name[4:]):

        output.append([line[0], line[1], line[2], line[3], value, name]) 
df = pd.DataFrame(output, columns=['study_id','participant', 'sex', 'species', 'value', 'condition_temp'])


In [7]:
df[['session','condition']] = df['condition_temp'].str.split('_playback_',expand=True)
df['session'].replace('first', '1', inplace=True)
df['session'].replace('second', '2', inplace=True)

complete_path_age = os.path.join(original_data_pathway, "subject_list.csv")
subject_list = pd.read_csv(complete_path_age)   
df= df.merge(subject_list,left_on='participant', right_on='name', how='left')
df.rename(columns={"age": "age_in_years"}, inplace=True)

df['year']='2014'

In [8]:

df=df[['study_id','year','participant', 'age_in_years','sex', 'species', 'session','condition', 'value']]


In [9]:

comp_out_path_stand = os.path.join(out_pathway, 'kano2016nasal_exp1_standardized.csv')
df.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


names =df.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
kano2016nasal_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'kano2016nasal_exp1_glossary.csv')
kano2016nasal_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
